In [23]:
from __future__ import annotations

import operator
import os
import re
from typing import TypedDict,List,Annotated,Literal
from pydantic import BaseModel,Field
from langgraph.graph import StateGraph,START,END
from langgraph.types import Send
from langchain_groq import ChatGroq
from langchain_core.messages import SystemMessage,HumanMessage
from pathlib import Path

# POSTGRES DATABASE
import psycopg
from langgraph.checkpoint.postgres import PostgresSaver
from psycopg.rows import dict_row
from dotenv import load_dotenv

load_dotenv()

True

In [24]:
def get_database_url():
    database_url = os.getenv("DATABASE_URL")

    if not database_url:
        raise ValueError(
            "DATABASE_URL is missing. Please add your Render PostgreSQL External Database URL to .env"
        )

    if "sslmode=" not in database_url:
        #If a query string already exists (has '?'), append with '&'. Otherwise, start it with '?'.
        separator = "&" if "?" in database_url else "?"
        database_url = f"{database_url}{separator}sslmode=require"

    # ? => marks beginning of query
    # & => separates multiple query parameters.If only one query parameter no need to use &
    # ex: https://example.com?parameter1=value1&parameter2=value2&parameter3=value3
    return database_url

GROQ_API_KEY = os.getenv("GROQ_API_KEY")
if not GROQ_API_KEY:
    raise ValueError("GROQ_API_KEY is missing. Please add it to your .env file.")

In [ ]:
class Task(BaseModel):
    id: int
    title: str
    goal: str = Field(...,
    description="One sentence describing what the reader should be able to do/understand after this section.",)
    bullets: List[str] = Field(...,min_length=3,max_length=5,
        description="3–5 concrete, non-overlapping subpoints to cover in this section.",)
    target_words: int = Field(...,description="Target word count for this section (20–45).",)
    section_type: Literal["intro", "core", "examples", "checklist", "common_mistakes", "conclusion"
    ] = Field(...,description="Use 'common_mistakes' exactly once in the plan.",)

class Plan(BaseModel):
    blog_title: str
    audience: str = Field(..., description="Who this blog is for.")
    tone: str = Field(..., description="Writing tone (e.g., practical, crisp).")
    tasks: List[Task]

class State(TypedDict):
    topic: str
    plan: Plan
    sections:Annotated[List[str],operator.add]  # reducer concatenates worker outputs
    final: str

llm = ChatGroq(model="openai/gpt-oss-20b",api_key=GROQ_API_KEY)

In [26]:
def orchestrator(state: State) -> dict:
    # print(state["sections"])
    planner = llm.with_structured_output(Plan)
    plan = planner.invoke(
        [
            SystemMessage(
                content=(
                    "You are a senior technical writer and developer advocate. Your job is to produce a "
                    "highly actionable outline for a technical blog post.\n\n"
                    "Hard requirements:\n"
                    "- Create 5–7 sections (tasks) that fit a technical blog.\n"
                    "- Each section must include:\n"
                    "  1) goal (1 sentence: what the reader can do/understand after the section)\n"
                    "  2) 3–5 bullets that are concrete, specific, and non-overlapping\n"
                    "  3) target word count (20–45)\n"
                    "- Include EXACTLY ONE section with section_type='common_mistakes'.\n\n"
                    "Make it technical (not generic):\n"
                    "- Assume the reader is a developer; use correct terminology.\n"
                    "- Prefer design/engineering structure: problem → intuition → approach → implementation → "
                    "trade-offs → testing/observability → conclusion.\n"
                    "- Bullets must be actionable and testable (e.g., 'Show a minimal code snippet for X', "
                    "'Explain why Y fails under Z condition', 'Add a checklist for production readiness').\n"
                    "- Explicitly include at least ONE of the following somewhere in the plan (as bullets):\n"
                    "  * a minimal working example (MWE) or code sketch\n"
                    "  * edge cases / failure modes\n"
                    "  * performance/cost considerations\n"
                    "  * security/privacy considerations (if relevant)\n"
                    "  * debugging tips / observability (logs, metrics, traces)\n"
                    "- Avoid vague bullets like 'Explain X' or 'Discuss Y'. Every bullet should state what "
                    "to build/compare/measure/verify.\n\n"
                    "Ordering guidance:\n"
                    "- Start with a crisp intro and problem framing.\n"
                    "- Build core concepts before advanced details.\n"
                    "- Include one section for common mistakes and how to avoid them.\n"
                    "- End with a practical summary/checklist and next steps.\n\n"
                    "Output must strictly match the Plan schema."
                )
            ),
            HumanMessage(content=f"Topic: {state['topic']}"),
        ]
    )
    return {"plan": plan}

def fanout(state: State):
    # print(state["sections"])
    return [Send("worker",{"task": task, "topic": state["topic"], "plan": state["plan"]},)
        for task in state["plan"].tasks]

def worker(payload: dict) -> dict:
    task = payload["task"]
    topic = payload["topic"]
    plan = payload["plan"]
    bullets_text = "\n- " + "\n- ".join(task.bullets)
    section_md = llm.invoke(
        [
            SystemMessage(
    content=(
        "You are a senior technical writer and developer advocate. Write ONE section of a technical blog post in Markdown.\n\n"
        "Hard constraints:\n"
        "- Follow the provided Goal and cover ALL Bullets in order (do not skip or merge bullets).\n"
        "- Stay close to the Target words (±15%).\n"
        "- Output ONLY the section content in Markdown (no blog title H1, no extra commentary).\n\n"
        "Technical quality bar:\n"
        "- Be precise and implementation-oriented (developers should be able to apply it).\n"
        "- Prefer concrete details over abstractions: APIs, data structures, protocols, and exact terms.\n"
        "- When relevant, include at least one of:\n"
        "  * a small code snippet (minimal, correct, and idiomatic)\n"
        "  * a tiny example input/output\n"
        "  * a checklist of steps\n"
        "  * a diagram described in text (e.g., 'Flow: A -> B -> C')\n"
        "- Explain trade-offs briefly (performance, cost, complexity, reliability).\n"
        "- Call out edge cases / failure modes and what to do about them.\n"
        "- If you mention a best practice, add the 'why' in one sentence.\n\n"
        "Markdown style:\n"
        "- Start with a '## <Section Title>' heading.\n"
        "- Use short paragraphs, bullet lists where helpful, and code fences for code.\n"
        "- Avoid fluff. Avoid marketing language.\n"
        "- If you include code, keep it focused on the bullet being addressed.\n"
    )
)
,
            HumanMessage(
                content=(
                    f"Blog: {plan.blog_title}\n"
                    f"Audience: {plan.audience}\n"
                    f"Tone: {plan.tone}\n"
                    f"Topic: {topic}\n\n"
                    f"Section: {task.title}\n"
                    f"Section type: {task.section_type}\n"
                    f"Goal: {task.goal}\n"
                    f"Target words: {task.target_words}\n"
                    f"Bullets:{bullets_text}\n"
                )
            ),
        ]
    ).content.strip()
    return {"sections": [section_md]}

def reducer(state: State) -> dict:
    print(state["sections"])
    title = state["plan"].blog_title
    body = "\n\n".join(state["sections"]).strip()
    final_md = f"# {title}\n\n{body}\n"
    # ---- save to file ----
    clean_title = re.sub(r'[^\w\s-]', '', title).strip().lower()
    filename = re.sub(r'[-\s]+', '_', clean_title) + ".md"

    Path(filename).write_text(final_md, encoding="utf-8")
    return {"final": final_md}

In [21]:
graph=StateGraph(State)

# add nodes
graph.add_node('orchestrator',orchestrator)
graph.add_node('worker',worker)
graph.add_node('reducer',reducer)

# add edges
graph.add_edge(START,'orchestrator')
graph.add_conditional_edges('orchestrator',fanout,['worker'])
graph.add_edge('worker','reducer')
graph.add_edge('reducer',END)

# =========================
# PostgreSQL Checkpointer
# =========================
DATABASE_URL = get_database_url()

_conn = psycopg.connect(
    DATABASE_URL,
    autocommit=True,
    row_factory=dict_row
)

checkpointer = PostgresSaver(_conn)
checkpointer.setup()


In [22]:
# Note: Whenever you wish to create a connection between agent and Database use checkpointer
# Pass the created checkpointer while compiling the graph
workflow=graph.compile(checkpointer=checkpointer)
# workflow

config = {"configurable": {"thread_id": "test_thread"}}
out = workflow.invoke({"topic": "Write a blog on Bellman Ford's Algorithm", "sections": []}, config=config)

['A linked list is a linear collection of nodes, each holding data and a reference to the next node, enabling dynamic memory usage and efficient insertions.', '## Types of Linked Lists  \nSingly, doubly, and circular lists differ in node links: a single forward link, bidirectional pointers, or wrap‑around. Use cases: simple queues, bidirectional traversal, and circular buffers.', '**Core Operations & Algorithms**\n\nInsertion, deletion, traversal, search, and reverse in linked lists—illustrated with concise C++, Java, and Python snippets.', '# Performance & Complexity Analysis\n\nLinked lists enable O(1) end insert/delete but O(n) random access; arrays give O(1) access, O(n) insertion, with lower space overhead.', '## Advanced Topics & Variants\n\nSkip lists accelerate search, while linked lists underpin efficient queues and stacks. Memory tricks like node pooling and compact pointers reduce overhead.', '## Practical Use Cases\n\nLinked lists power OS process queues, network packet buf